In [1]:
import napari
from aicsimageio import AICSImage
import numpy as np
from tifffile import imwrite
import os

26-Nov-25 11:29:22 - bfio.backends - WARNING  - Java backend is not available. This could be due to a missing dependency (jpype).


In [2]:


def leica_loader(path_file, visualize = False):
    # Load the .lif file
    path_data = "/home/gerard/data/confocal"
    img = AICSImage("/home/gerard/data/confocal/2025_01_23_Darryl/Project.lif")

    # Extract the numpy array (use `get_image_data()` or `data` attribute)
    image_data = img.get_image_data()  # OR `img.data`

    # Print shape to verify
    print(f"Image shape: {image_data.shape}")
    
    if visualize:
        # Open in Napari
        viewer = napari.Viewer()
        viewer.add_image(image_data)  # Pass the numpy array, not the AICSImage object
        napari.run()

In [ ]:
    # Load the .lif file
def channel_split(date):
    """_summary_

    Args:
        date (str): example: '2025_01_23'
    """
   
    
    project_path = os.path.join('/home/gerard/data/confocal', f"{date}_Darryl")
    
    img = AICSImage(os.path.join(project_path, 'Project.lif'))
    
    # series_count = img.reader.series_count
    # print(f"The file has {series_count} series")
    
    print("Number of series:", img.reader.dask_data.shape[0])  # Or use reader.series_count
    
    print(f"type: {type(img)} dimensions:{img.shape}")
    # Extract the numpy array (use `get_image_data()` or `data` attribute)
    image_data = img.get_image_data()  # OR `img.data`
    print(f"changed to {type(image_data)}")
    print(image_data.shape)
    print(f"number of channels : {image_data.shape[1]}")
    for channel in range(image_data.shape[1]):
        
        output_path = os.path.join(project_path, f"channel_{channel}")
        os.makedirs(output_path, exist_ok=True)
        for frame in range(image_data.shape[2]):
            f = image_data[0, channel, frame, :, :]
            name = f"{date}_ch{channel}_f{frame}.tif"
            imwrite(os.path.join(output_path, name), f)
            
            

        



In [4]:
from aicsimageio.readers import LifReader

def loader2(date,user,split_frames=False):
    
    """_summary_
    
    """
    
    all_path = os.path.join('/home/gerard/data/confocal', f"{date}_{user}")
    project_path = os.path.join(all_path, 'Project.lif')
    reader = LifReader(project_path)
    print(f"Number of series: {len(reader.scenes)}")
    
    for i, series in enumerate(reader.scenes):
        series_path = os.path.join(all_path, f"series_{i}")
        os.makedirs(series_path, exist_ok=True)
        reader.set_scene(series)
        print(f"series {i}: shape = {reader.dims.shape}")
        num_channels = reader.dims.C 
        
        for channel in range(0,num_channels):

            if split_frames == False:
                zyx = reader.get_image_data("ZYX", T=0, C=channel)
                name_c = f"{date}_s{i}_ch{channel}.tif"
                imwrite(os.path.join(series_path, name_c), zyx)
            else:
                channel_path = os.path.join(series_path,f"channel_{channel}" )
                os.makedirs(channel_path, exist_ok=True)
                num_frames = reader.dims.Z
                for frame in range(0,num_frames):
                    yx = reader.get_image_data("YX", T=0, C=channel, Z=frame)
                    name_f = f"{date}_s{i}_ch{channel}_f{frame}.tif"
                    imwrite(os.path.join(channel_path, name_f), yx)

                    
                
                
        
    

In [8]:
# channel_split('2025_04_07')
loader2('2025_11_24','Gerardo',True)

Number of series: 4
series 0: shape = (1, 1, 1, 512, 512)
series 1: shape = (1, 1, 1, 512, 512)
series 2: shape = (1, 3, 111, 512, 512)
series 3: shape = (1, 3, 110, 512, 512)


**jigkhvhjv**
